In [20]:
!pip install -q openai tqdm pandas

In [21]:
import os
import re
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from google.colab import userdata
from openai import OpenAI

# 1. Fetch API key from Colab Secrets
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    raise ValueError("Key 'OPENROUTER_API_KEY' not found in Colab Secrets.") from e

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# 2. Dosage Evaluation Prompts
SYSTEM_PROMPT = (
    "You are a strict binary evaluator. Return ONLY the single digit 1 or 0 as your final output. "
    "Do not output reasoning, commentary, or formatting."
)

USER_PROMPT_TEMPLATE = """Sentence (Ground Truth): {text}
Target Dosage: {target_dosage}

Task:
Determine if the target_dosage is correctly present and matched in the sentence.
- Output 1 if target_dosage is correct and present in the sentence.
- Output 0 if target_dosage is hallucinated.

Output strictly 1 or 0:"""

# 3. Core Evaluation Logic
def evaluate_pair(text, target_dosage, model_name, max_retries=3):
    if pd.isna(text) or pd.isna(target_dosage) or not str(text).strip() or not str(target_dosage).strip():
        return 0

    user_prompt = USER_PROMPT_TEMPLATE.format(
        text=str(text).strip(),
        target_dosage=str(target_dosage).strip()
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0,
                max_tokens=3
            )

            if response and response.choices and len(response.choices) > 0:
                choice = response.choices[0]
                if choice.message and choice.message.content is not None:
                    content = choice.message.content.strip()
                    match = re.search(r'[01]', content)
                    if match:
                        return int(match.group(0))

            time.sleep(1 * (attempt + 1))

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"\n[API Error - {model_name}]: {e}")
            time.sleep(1 * (attempt + 1))

    return 0

# 4. Driver function to process a single model, generate CSV, and print summary
def run_model_evaluation(display_name, model_id, input_file, output_file, max_workers=5, save_every=50):
    print("\n" + "="*60)
    print(f"   STARTING EVALUATION FOR: {display_name}")
    print("="*60)

    if os.path.exists(output_file):
        print(f"Resuming from existing output file '{output_file}'...")
        df = pd.read_csv(output_file)
    else:
        df = pd.read_csv(input_file)

    if 'correct_pair' not in df.columns:
        df['correct_pair'] = None
    if 'incorrect_pair' not in df.columns:
        df['incorrect_pair'] = None

    df['correct_pair'] = df['correct_pair'].astype(object)
    df['incorrect_pair'] = df['incorrect_pair'].astype(object)

    def process_row(idx, row):
        res_correct = row['correct_pair']
        res_incorrect = row['incorrect_pair']

        if pd.isna(res_correct) or str(res_correct).strip() == "":
            res_correct = evaluate_pair(row['text'], row['Dosage'], model_id)

        if pd.isna(res_incorrect) or str(res_incorrect).strip() == "":
            res_incorrect = evaluate_pair(row['modified_text'], row['Dosage'], model_id)

        return idx, int(res_correct), int(res_incorrect)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_row, idx, row) for idx, row in df.iterrows()]

        completed_count = 0
        for future in tqdm(as_completed(futures), total=len(futures), desc=display_name):
            idx, res_correct, res_incorrect = future.result()
            df.at[idx, 'correct_pair'] = res_correct
            df.at[idx, 'incorrect_pair'] = res_incorrect

            completed_count += 1
            if completed_count % save_every == 0:
                df.to_csv(output_file, index=False, encoding='utf-8-sig')

    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n✅ Output CSV saved to: '{output_file}'")

    # Metrics Summary Print
    total_rows = len(df)
    correct_acc = (df['correct_pair'] == 1).sum() / total_rows * 100
    incorrect_acc = (df['incorrect_pair'] == 0).sum() / total_rows * 100
    overall_acc = ((df['correct_pair'] == 1).sum() + (df['incorrect_pair'] == 0).sum()) / (2 * total_rows) * 100

    print("\n" + "-"*45)
    print(f"       SUMMARY: {display_name.upper()}       ")
    print("-"*45)
    print(f"Total Evaluated Rows        : {total_rows}")
    print(f"Correct Pair Accuracy (1s)   : {correct_acc:.2f}%")
    print(f"Incorrect Pair Accuracy (0s) : {incorrect_acc:.2f}%")
    print(f"Overall Accuracy            : {overall_acc:.2f}%")
    print("-"*45 + "\n")

In [22]:
run_model_evaluation(
    display_name="GPT-4o-mini",
    model_id="openai/gpt-4o-mini",
    input_file="transformed_with_luna_modified_dosage_cleaned.csv",
    output_file="evaluated_gpt_4o_mini_dosage.csv"
)


   STARTING EVALUATION FOR: GPT-4o-mini


GPT-4o-mini: 100%|██████████| 495/495 [02:23<00:00,  3.45it/s]


✅ Output CSV saved to: 'evaluated_gpt_4o_mini_dosage.csv'

---------------------------------------------
       SUMMARY: GPT-4O-MINI       
---------------------------------------------
Total Evaluated Rows        : 495
Correct Pair Accuracy (1s)   : 99.60%
Incorrect Pair Accuracy (0s) : 82.22%
Overall Accuracy            : 90.91%
---------------------------------------------



In [23]:
run_model_evaluation(
    display_name="Gemini-2.5-flash-lite",
    model_id="google/gemini-2.5-flash-lite",
    input_file="transformed_with_luna_modified_dosage_cleaned.csv",
    output_file="evaluated_gemini_2_5_flash_lite_dosage.csv"
)


   STARTING EVALUATION FOR: Gemini-2.5-flash-lite


Gemini-2.5-flash-lite: 100%|██████████| 495/495 [01:23<00:00,  5.91it/s]


✅ Output CSV saved to: 'evaluated_gemini_2_5_flash_lite_dosage.csv'

---------------------------------------------
       SUMMARY: GEMINI-2.5-FLASH-LITE       
---------------------------------------------
Total Evaluated Rows        : 495
Correct Pair Accuracy (1s)   : 99.19%
Incorrect Pair Accuracy (0s) : 87.27%
Overall Accuracy            : 93.23%
---------------------------------------------



In [24]:
run_model_evaluation(
    display_name="Qwen 2.5 72B Instruct",
    model_id="qwen/qwen-2.5-72b-instruct",
    input_file="transformed_with_luna_modified_dosage_cleaned.csv",
    output_file="evaluated_qwen_2_5_72b_instruct_dosage.csv"
)


   STARTING EVALUATION FOR: Qwen 2.5 72B Instruct


Qwen 2.5 72B Instruct: 100%|██████████| 495/495 [01:21<00:00,  6.08it/s]


✅ Output CSV saved to: 'evaluated_qwen_2_5_72b_instruct_dosage.csv'

---------------------------------------------
       SUMMARY: QWEN 2.5 72B INSTRUCT       
---------------------------------------------
Total Evaluated Rows        : 495
Correct Pair Accuracy (1s)   : 99.80%
Incorrect Pair Accuracy (0s) : 81.01%
Overall Accuracy            : 90.40%
---------------------------------------------



In [25]:
run_model_evaluation(
    display_name="Llama 3.1 70B Instruct",
    model_id="meta-llama/llama-3.1-70b-instruct",
    input_file="transformed_with_luna_modified_dosage_cleaned.csv",
    output_file="evaluated_llama_3_1_70b_instruct_dosage.csv"
)


   STARTING EVALUATION FOR: Llama 3.1 70B Instruct


Llama 3.1 70B Instruct: 100%|██████████| 495/495 [01:42<00:00,  4.84it/s]


✅ Output CSV saved to: 'evaluated_llama_3_1_70b_instruct_dosage.csv'

---------------------------------------------
       SUMMARY: LLAMA 3.1 70B INSTRUCT       
---------------------------------------------
Total Evaluated Rows        : 495
Correct Pair Accuracy (1s)   : 98.79%
Incorrect Pair Accuracy (0s) : 50.91%
Overall Accuracy            : 74.85%
---------------------------------------------



In [26]:
run_model_evaluation(
    display_name="DeepSeek V3 Flash",
    model_id="deepseek/deepseek-chat",
    input_file="transformed_with_luna_modified_dosage_cleaned.csv",
    output_file="evaluated_deepseek_v3_flash_dosage.csv"
)


   STARTING EVALUATION FOR: DeepSeek V3 Flash


DeepSeek V3 Flash: 100%|██████████| 495/495 [03:06<00:00,  2.66it/s]


✅ Output CSV saved to: 'evaluated_deepseek_v3_flash_dosage.csv'

---------------------------------------------
       SUMMARY: DEEPSEEK V3 FLASH       
---------------------------------------------
Total Evaluated Rows        : 495
Correct Pair Accuracy (1s)   : 100.00%
Incorrect Pair Accuracy (0s) : 85.45%
Overall Accuracy            : 92.73%
---------------------------------------------



In [27]:
import os
import re
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from google.colab import userdata
from openai import OpenAI

# 1. API Setup
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    raise ValueError("Key 'OPENROUTER_API_KEY' not found in Colab Secrets.") from e

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# Configuration
MODEL_NAME = "deepseek/deepseek-chat"
DISPLAY_NAME = "DeepSeek V3 Flash"
INPUT_FILE = "transformed_with_luna_modified_dosage_cleaned.csv"
OUTPUT_FILE = "evaluated_deepseek_v3_flash_dosage.csv"

MAX_WORKERS = 4          # Slightly lower worker count to mitigate rate limits/dropouts
MAX_RETRIES = 3          # Attempts per individual call
MAX_LOOP_ATTEMPTS = 6    # Max retry passes across the whole dataset
FAILURE_THRESHOLD = 5    # Stop rerunning when total unresolved API calls <= 5

# Prompts
SYSTEM_PROMPT = (
    "You are a strict binary evaluator. Return ONLY the single digit 1 or 0 as your final output. "
    "Do not output reasoning, commentary, or formatting."
)

USER_PROMPT_TEMPLATE = """Sentence (Ground Truth): {text}
Target Dosage: {target_dosage}

Task:
Determine if the target_dosage is correctly present and matched in the sentence.
- Output 1 if target_dosage is correct and present in the sentence.
- Output 0 if target_dosage is hallucinated.

Output strictly 1 or 0:"""

def evaluate_pair(text, target_dosage, max_retries=MAX_RETRIES):
    """Evaluates a pair. Returns 1, 0, or None if the API call explicitly fails."""
    if pd.isna(text) or pd.isna(target_dosage) or not str(text).strip() or not str(target_dosage).strip():
        return 0

    user_prompt = USER_PROMPT_TEMPLATE.format(
        text=str(text).strip(),
        target_dosage=str(target_dosage).strip()
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0,
                max_tokens=3
            )

            if response and response.choices and len(response.choices) > 0:
                choice = response.choices[0]
                if choice.message and choice.message.content is not None:
                    content = choice.message.content.strip()
                    match = re.search(r'[01]', content)
                    if match:
                        return int(match.group(0))

            time.sleep(1 * (attempt + 1))

        except Exception as e:
            time.sleep(1.5 * (attempt + 1))

    # Return None on genuine API/network failure so it can be retried
    return None


# Load input data
df = pd.read_csv(INPUT_FILE)
df['correct_pair'] = None
df['incorrect_pair'] = None

loop_count = 0

while loop_count < MAX_LOOP_ATTEMPTS:
    loop_count += 1

    # Identify pending/failed pairs
    missing_correct = df['correct_pair'].isna().sum()
    missing_incorrect = df['incorrect_pair'].isna().sum()
    total_failures = missing_correct + missing_incorrect

    print(f"\n--- Pass {loop_count}/{MAX_LOOP_ATTEMPTS} ---")
    print(f"Unresolved API Failures: {total_failures} (Correct pair: {missing_correct}, Incorrect pair: {missing_incorrect})")

    if total_failures <= FAILURE_THRESHOLD:
        print(f"✅ Success! Unresolved failures ({total_failures}) are within threshold (<= {FAILURE_THRESHOLD}).")
        break

    def process_row(idx, row):
        res_correct = row['correct_pair']
        res_incorrect = row['incorrect_pair']

        if pd.isna(res_correct):
            res_correct = evaluate_pair(row['text'], row['Dosage'])

        if pd.isna(res_incorrect):
            res_incorrect = evaluate_pair(row['modified_text'], row['Dosage'])

        return idx, res_correct, res_incorrect

    # Submit only incomplete rows
    rows_to_process = [
        (idx, row) for idx, row in df.iterrows()
        if pd.isna(row['correct_pair']) or pd.isna(row['incorrect_pair'])
    ]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_row, idx, row) for idx, row in rows_to_process]

        completed = 0
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Evaluating DeepSeek (Pass {loop_count})"):
            idx, res_correct, res_incorrect = future.result()
            df.at[idx, 'correct_pair'] = res_correct
            df.at[idx, 'incorrect_pair'] = res_incorrect
            completed += 1
            if completed % 25 == 0:
                df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    time.sleep(2)  # Cooldown between pass attempts

# Save final CSV output
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"\n✅ Rerun complete! Output saved to '{OUTPUT_FILE}'.")

# Generate summary metrics
total_rows = len(df)
remaining_failures = df['correct_pair'].isna().sum() + df['incorrect_pair'].isna().sum()

valid_correct = df['correct_pair'].dropna()
valid_incorrect = df['incorrect_pair'].dropna()

correct_acc = (valid_correct == 1).sum() / total_rows * 100
incorrect_acc = (valid_incorrect == 0).sum() / total_rows * 100
overall_acc = ((valid_correct == 1).sum() + (valid_incorrect == 0).sum()) / (2 * total_rows) * 100

print("\n" + "-"*45)
print(f"       SUMMARY: {DISPLAY_NAME.upper()}       ")
print("-"*45)
print(f"Total Evaluated Rows        : {total_rows}")
print(f"Remaining Unresolved Calls  : {remaining_failures}")
print(f"Correct Pair Accuracy (1s)   : {correct_acc:.2f}%")
print(f"Incorrect Pair Accuracy (0s) : {incorrect_acc:.2f}%")
print(f"Overall Accuracy            : {overall_acc:.2f}%")
print("-"*45 + "\n")


--- Pass 1/6 ---
Unresolved API Failures: 990 (Correct pair: 495, Incorrect pair: 495)


Evaluating DeepSeek (Pass 1): 100%|██████████| 495/495 [03:40<00:00,  2.24it/s]



--- Pass 2/6 ---
Unresolved API Failures: 0 (Correct pair: 0, Incorrect pair: 0)
✅ Success! Unresolved failures (0) are within threshold (<= 5).

✅ Rerun complete! Output saved to 'evaluated_deepseek_v3_flash_dosage.csv'.

---------------------------------------------
       SUMMARY: DEEPSEEK V3 FLASH       
---------------------------------------------
Total Evaluated Rows        : 495
Remaining Unresolved Calls  : 0
Correct Pair Accuracy (1s)   : 100.00%
Incorrect Pair Accuracy (0s) : 83.03%
Overall Accuracy            : 91.52%
---------------------------------------------

